In [0]:
# Section 1 — Read Bronze Product Table

df = spark.table("bike_lakehouse.bronze.crm_prd_info")

display(df)

In [0]:

# Section 2 — Inspect Product Schema

df.printSchema()


In [0]:
# Section 2 — Check NULL Values

from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
# Section 2 — Inspect Missing Product Attributes

display(
    df.filter(
        col("prd_cost").isNull() |
        col("prd_line").isNull()
    )
)

In [0]:
# Section 3 — Clean Product Strings

from pyspark.sql.functions import trim

df_clean = (
    df
    .withColumn("prd_nm", trim(col("prd_nm")))
    .withColumn("prd_line", trim(col("prd_line")))
)

display(df_clean.limit(20))

In [0]:
# Section 4 — Product Quality Check

print("Total rows:", df_clean.count())

print("NULL product IDs:", df_clean.filter(col("prd_id").isNull()).count())

print("NULL product keys:", df_clean.filter(col("prd_key").isNull()).count())

print("NULL product names:", df_clean.filter(col("prd_nm").isNull()).count())

print("NULL start dates:", df_clean.filter(col("prd_start_dt").isNull()).count())

In [0]:
# Section 5 — Friendly Column Names

df_silver = (
    df_clean
    .withColumnRenamed("prd_id", "product_id")
    .withColumnRenamed("prd_key", "product_key")
    .withColumnRenamed("prd_nm", "product_name")
    .withColumnRenamed("prd_cost", "product_cost")
    .withColumnRenamed("prd_line", "product_line")
    .withColumnRenamed("prd_start_dt", "start_date")
    .withColumnRenamed("prd_end_dt", "end_date")
)

display(df_silver.limit(20))

In [0]:
# Section 6 — Write Silver Product Table

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bike_lakehouse.silver.crm_products")

In [0]:
# Section 7 — Final Verification

spark.sql("""
SELECT COUNT(*) AS row_count
FROM bike_lakehouse.silver.crm_products
""").show()

spark.sql("""
DESCRIBE bike_lakehouse.silver.crm_products
""").show()